# Script in which we study the properties of the 18 isolated galaxies in voids with presence of AGN

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
#Salim, catalog GSWCL-A2 (for sSFR)
Salim_A = pd.read_csv('data/Salim_isa.csv') #this calatog contains every sSFR for our galaxy sample


Pann_SIG = pd.read_csv('data/Pann_SIG.csv')
cond2 = Pann_SIG['Q_LSS']>-20


SIG_AGN_walls = pd.read_csv('data/SIG_AGN_walls.csv')
Pann_SIG_AGN = pd.read_csv('data/Pann_SIG_AGN.csv')

Pann_SIG

The most isolated galaxies in our sample are :

> 1237662198826139791 ; con TType = 0.050386, Q_lss = -7.425087, dnn = 4.600468


> 1237658298996097265 ; con TType = -1.641851, Q_lss = -7.028068,	dnn = 4.079681

In [ ]:
Salim_A #This contains the data for sSFR

Lacerna et.al (2014) separation criteria: 

> For the SF we have the following relation to separate. Above is SF and below is Quenched:
 '''
 log_{10}(sSFR) = -0.6173·log_{10}(M_{*}) - 4.72
 '''

> For the color. Above is red and below is blue:
 '''
 g - i = 0.2006· \log(M_s) - 1.02
 '''

> For the morphology we have. TType<0 Early type and in Late type we have another partition with:
 '''
 TType = -0.962· \log(M_s) + 12.30 \quad (TType>0)
 '''

## Functions

In [ ]:

#I'll define the functions separately with the criteria I know for convenience

#dataframe I'm using: 
x_mass = np.linspace(8.5,12,100)


#MORPHOLOGY (TType)

y_morpho = -0.962*x_mass + 12.30 

#COLOR (red/blue)

y_color = 0.2006*x_mass - 1.02

#STAR FORMING RATE (SF/quenched)

y_sSfr = -0.6173*x_mass - 4.72



### Specific star forming rate (data from https://salims.pages.iu.edu/gswlc/)

In [ ]:

Pann_SIG_sfr_merge = pd.merge (Pann_SIG, Salim_A, on= ['objID'], how= 'left') #this one has the data of Pann-SIG with sSFR

Pann_SIG_sfr = Pann_SIG_sfr_merge.dropna(subset=['log(sSFR)']).reset_index(drop=True)

Pann_SIG_sfr.to_csv('data/Pann_SIG_ALL.csv') #this one contains the isolated galaxies in voids with all the data available

Pann_SIG_sfr

> REMINDER: sSFR is in logaritmic scale

In [ ]:

cond2 = cond2.reindex(Pann_SIG_sfr.index) #to equal the boolean series' cond2 index to that of df Pann_SIG_sfr
cond2 = cond2.fillna(False) #to avoid empty indeces to be filled with a Nan value

sSfr_err = 3*0.1189 #this will allow me to plot the green valley

fig, ax = plt.subplots()

ax.plot(x_mass, y_sSfr, linestyle = '-', label = 'SF/Q')
ax.fill_between(x_mass, y_sSfr + sSfr_err, y_sSfr - sSfr_err, color = 'blue', alpha = 0.2, label = r'$\pm2\,\sigma$')


x = Pann_SIG_sfr[cond2]['logM*']
y = Pann_SIG_sfr[cond2]['log(sSFR)']

x_A = Pann_SIG_AGN['stellarmass']
y_A = Pann_SIG_AGN['log(sSFR)']


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s = 6)
plt.scatter(x_A, y_A, c=Pann_SIG_AGN['dnn'], marker='o', label='SIG AGN in voids', vmin=1, vmax=5)

cbar = plt.colorbar()
cbar.set_label(r'$\rm dnn (Mpc) $', fontsize = 12 )

plt.ylim(-13.5, -9.2)
plt.xlim(8.1, 12.2)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm log(sSFR / ysr^{-1}) $', fontsize = 14 ) 
plt.tick_params( axis= 'both', labelsize = 12)

plt.legend()
plt.show

plt.savefig('plots/AGN_voids_sfr_dnn.png')

In [ ]:

cond2 = cond2.reindex(Pann_SIG_sfr.index) 
cond2 = cond2.fillna(False) 

sSfr_err = 3*0.1189 

fig, ax = plt.subplots()

ax.plot(x_mass, y_sSfr, linestyle = '-', label = 'SF/Q')
ax.fill_between(x_mass, y_sSfr + sSfr_err, y_sSfr - sSfr_err, color = 'blue', alpha = 0.2, label = r'$\pm2\,\sigma$')


x = Pann_SIG_sfr[cond2]['logM*']
y = Pann_SIG_sfr[cond2]['log(sSFR)']

x_A = np.array(Pann_SIG_AGN['stellarmass'])
y_A = np.array(Pann_SIG_AGN['log(sSFR)'])

p_bar = np.array(Pann_SIG_AGN['P_bar_GZ2'])


x_E = [x_A[i] for i in sorted(range(len(x_A)), key = lambda k: p_bar[k])]
y_E  = [y_A[i] for i in sorted(range(len(y_A)), key = lambda k: p_bar[k])]


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s = 6)
plt.scatter(x_A, y_A, c=sorted(Pann_SIG_AGN['P_bar_GZ2']), marker='o', label='SIG AGN in voids',vmin=0, vmax=1)

cbar = plt.colorbar()
cbar.set_label(r'$\rm P _{bar} $', fontsize = 12 )

#plt.title('Star forming rate fit for AGN isolated galaxies in voids')
plt.ylim(-13.5, -9.2) #there are two galaxies with sSFR=0, I have asked it to not read them
plt.xlim(8.1, 12.2)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm log(sSFR / ysr^{-1} )$', fontsize = 14 ) 
plt.tick_params( axis= 'both', labelsize = 12)

plt.legend()
plt.show

plt.savefig('plots/AGN_voids_sfr_pbar.png')

### Color

In [ ]:
color_err = 0.045668158646274284

fig, ax = plt.subplots()

ax.plot(x_mass, y_color, linestyle = '-', label = 'R/B') #This does not have a green valley

Pann_SIG['g-i'] = Pann_SIG['MAG_g'] - Pann_SIG['MAG_i']

x = Pann_SIG[cond2]['stellarmass']
y = Pann_SIG[cond2]['g-i']

x_B = Pann_SIG_AGN['stellarmass']
y_B = Pann_SIG_AGN['g-i']


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s=6)
plt.scatter(x_B, y_B, c=Pann_SIG_AGN['dnn'], marker='o', label='SIG AGN in voids', vmin=1, vmax=5)

cbar = plt.colorbar()
cbar.set_label(r'$\rm dnn (Mpc) $', fontsize = 12)



#plt.title('Color fit for AGN isolated galaxies in voids')
plt.ylim(0.2, 1.65) 
plt.xlim(8.2, 12.1)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm g-i $', fontsize = 14 ) 

plt.tick_params( axis= 'both', labelsize = 12)

plt.legend()
plt.show

plt.savefig('plots/AGN_voids_color_dnn.png')

In [ ]:
color_err = 0.045668158646274284

fig, ax = plt.subplots()

ax.plot(x_mass, y_color, linestyle = '-', label = 'R/B')

Pann_SIG['g-i'] = Pann_SIG['MAG_g'] - Pann_SIG['MAG_i']

x = Pann_SIG[cond2]['stellarmass']
y = Pann_SIG[cond2]['g-i']

x_B = Pann_SIG_AGN['stellarmass']
y_B = Pann_SIG_AGN['g-i']


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s=6)
plt.scatter(x_B, y_B, c=Pann_SIG_AGN['P_bar_GZ2'], marker='o', label='SIG AGN in voids', vmin=0, vmax=1)
#in this case the colors represent the probability of founding a bar in that galaxy

cbar = plt.colorbar()
cbar.set_label(r'$\rm P_{bar} $', fontsize = 13)


plt.ylim(0.2, 1.65) 
plt.xlim(8.2, 12.1)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm g-i $', fontsize = 14 ) 

plt.tick_params( axis= 'both', labelsize = 12)

plt.legend()
plt.show

plt.savefig('plots/AGN_voids_color_pbar.png')

### Morfología

In [ ]:

morpho_err = 0.25700983622416396

fig, ax = plt.subplots()

plt.axhline( y = 0, color = 'b', linestyle = '-', label = 'LT/ET')

#I am not including condition this time because I am interested in the negative values of TType too

x = Pann_SIG[cond2]['stellarmass']
y = Pann_SIG[cond2]['TType_DS']

x_C = Pann_SIG_AGN['stellarmass']
y_C = Pann_SIG_AGN['TType']


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s = 6)
plt.scatter(x_C, y_C, c=Pann_SIG_AGN['dnn'], marker='o', label='SIG AGN in voids', vmin=1, vmax=5)
#in this case the colors represent the probability of founding a bar in that galaxy

cbar = plt.colorbar()
cbar.set_label(r'$\rm dnn (Mpc) $', fontsize = 12 )
plt.tick_params( axis= 'both', labelsize = 12)


#plt.title('Morphology fit for AGN isolated galaxies in voids')
#plt.ylim(-13, -8)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm TType $', fontsize = 14 ) 

plt.legend()
plt.show

#One of my super isolated galaxies is early type, and the other one is late type

plt.savefig('plots/AGN_voids_morpho_dnn.png')


In [ ]:

morpho_err = 0.25700983622416396

fig, ax = plt.subplots()

plt.axhline( y = 0, color = 'b', linestyle = '-', label = 'LT/ET')

x = Pann_SIG[cond2]['stellarmass']
y = Pann_SIG[cond2]['TType_DS']

x_C = Pann_SIG_AGN['stellarmass']
y_C = Pann_SIG_AGN['TType']


plt.scatter(x, y, color='grey', marker='o', label='SIG in voids', alpha = 0.4, s = 6)
plt.scatter(x_C, y_C, c=Pann_SIG_AGN['P_bar_GZ2'], marker='o', label='SIG AGN in voids', vmin=0, vmax=1) 
#in this case the colors represent the probability of founding a bar in that galaxy

cbar = plt.colorbar()
cbar.set_label(r'$\rm P_{bar} $', fontsize = 13 )
plt.tick_params( axis= 'both', labelsize = 12)


#plt.title('Morphology fit for AGN isolated galaxies in voids')
#plt.ylim(-13, -8)
plt.xlabel(r'$\rm log ( M_\star / M_\odot) $', fontsize = 14 ) 
plt.ylabel(r'$\rm TType $', fontsize = 14 ) 

plt.legend()
plt.show

plt.savefig('plots/AGN_voids_morpho_pbar.png')